# Lab 6 - Ecuación normal — Solución

Solución: cálculo de `heta` por ecuación normal sobre datos sintéticos (equivalentes al laboratorio anterior) y aplicación multivariante al dataset de Boston.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
np.random.seed(0)

## 1) Aplicar ecuación normal a datos (datos equivalentes al laboratorio anterior)

In [2]:
# Generar datos lineales multivariados (equivalente al laboratorio previo)
N = 200
x1 = 2 * np.random.rand(N) - 1
x2 = 2 * np.random.rand(N) - 1
theta_true = np.array([0.5, 2.1, -3.1])  # θ0, θ1, θ2
y = theta_true[0] + theta_true[1]*x1 + theta_true[2]*x2 + np.random.randn(N)*0.1
# Diseño (N, n_features) con término independiente
X_design = np.column_stack([np.ones(N), x1, x2])
# Vector objetivo
y_vec = y.reshape(-1)
# Ecuación normal con control de condición
XtX = X_design.T @ X_design
cond_number = np.linalg.cond(XtX)
print('Condición de X^T X:', cond_number)
if cond_number < 1e12:
    theta_normal = np.linalg.inv(XtX) @ X_design.T @ y_vec
else:
    lam = 1e-6
    theta_normal = np.linalg.inv(XtX + lam * np.eye(XtX.shape[0])) @ X_design.T @ y_vec
print('θ (ecuación normal):', theta_normal)
print('θ verdaderos      :', theta_true)
from IPython.display import display, Markdown
display(Markdown(r'''
Ecuación normal (sin regularización):
$$\Theta = (X^T X)^{-1} X^T y$$
Si $X^T X$ está mal condicionada, se usa Ridge:
$$\Theta = (X^T X + \lambda I)^{-1} X^T y$$
'''))

Condición de X^T X: 3.2024523211981606
θ (ecuación normal): [ 0.48954099  2.11279697 -3.09895141]
θ verdaderos      : [ 0.5  2.1 -3.1]



Ecuación normal (sin regularización):
$$\Theta = (X^T X)^{-1} X^T y$$
Si $X^T X$ está mal condicionada, se usa Ridge:
$$\Theta = (X^T X + \lambda I)^{-1} X^T y$$


## 2) Regresión multivariante sobre Boston (ecuación normal)

In [3]:
# Cargar el dataset de Boston desde la URL proporcionada en el enunciado
import urllib.request
url = 'http://lib.stat.cmu.edu/datasets/boston'
raw = urllib.request.urlopen(url).read().decode('utf-8').splitlines()
data_lines = raw[22:]
vals = []
for i in range(0, len(data_lines), 2):
    a = data_lines[i].strip().split()
    b = data_lines[i+1].strip().split()
    vals.append(a + b)
arr = np.array(vals, dtype=float)
# Reconstrucción conforme al snippet: las características están en 'data' y la columna 2 del segundo bloque es el target
# Aquí construimos X con todas las columnas disponibles (multivariante)
# data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]]) equivalencia:
# arr contiene todas las columnas concatenadas por filas; convertimos a diseño
# En esta fuente, las columnas útiles (según el enunciado) ocupan las posiciones de arr
# Construimos X_design_boston con intercept y todas columnas de arr excepto la columna target (col 2 of second row)
# Para seguridad, usaremos la forma estándar usada en el curso:
raw_df = pd.read_csv(url, sep='\s+', skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]
# data tiene muchas columnas; usamos todas ellas como características
X_b = data
y_b = target
# Diseño con intercept
Xb_design = np.column_stack([np.ones(X_b.shape[0]), X_b])
# Ecuación normal multivariante
XtX_b = Xb_design.T @ Xb_design
cond_b = np.linalg.cond(XtX_b)
print('Condición X^T X (Boston):', cond_b)
if cond_b < 1e12:
    theta_b = np.linalg.inv(XtX_b) @ Xb_design.T @ y_b
else:
    lam = 1.0
    theta_b = np.linalg.inv(XtX_b + lam * np.eye(XtX_b.shape[0])) @ Xb_design.T @ y_b
print('Dim theta_b:', theta_b.shape)
# Comparar con sklearn (tarda poco)
lr = LinearRegression()
lr.fit(Xb_design, y_b)
theta_sklearn = np.hstack([lr.intercept_, lr.coef_])
print('Norma diferencia (manual vs sklearn):', np.linalg.norm(theta_b - theta_sklearn))
# Predicción y R2
y_pred = Xb_design @ theta_b
print('R2 (manual):', r2_score(y_b, y_pred))
print('R2 (sklearn):', lr.score(Xb_design, y_b))
# Visualizar predicho vs real (muestra)
plt.figure(figsize=(6,4))
plt.scatter(y_b, y_pred, alpha=0.6, s=10)
mn = min(y_b.min(), y_pred.min())
mx = max(y_b.max(), y_pred.max())
plt.plot([mn,mx],[mn,mx],'r--')
plt.xlabel('Real')
plt.ylabel('Predicho')
plt.title('Boston: Predicho vs Real (ecuación normal multivariante)')
plt.show()

<>:19: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:19: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
/tmp/ipykernel_12744/1268830749.py:19: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  raw_df = pd.read_csv(url, sep='\s+', skiprows=22, header=None)


Condición X^T X (Boston): 228418414.2210897
Dim theta_b: (14,)


/tmp/ipykernel_12744/1268830749.py:19: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  raw_df = pd.read_csv(url, sep='\s+', skiprows=22, header=None)


ValueError: operands could not be broadcast together with shapes (14,) (15,) 

---
### Observaciones
- La ecuación normal da la solución cerrada: $\Theta=(X^T X)^{-1}X^T y$.
- Si $X^T X$ está mal condicionada se debe usar regularización (Ridge) para estabilizar la inversa.
- Se compara la solución manual con `sklearn` para validar resultados.